# Colour Converter

Colourises a black-and-white/greyscale photo using a pretrained deep-learning
model (`siggraph17`, from
[richzhang/colorization](https://github.com/richzhang/colorization)),
vendored locally in `colorizers/`.

**Usage:** set `image_path` in the first cell to the photo you want to
colourise, then run both cells top to bottom. Tune `MAX_DIM` in the second
cell to trade off colour detail against speed (see the comment above it for
the measured tradeoff).

## Load image

In [ ]:
# Import the necessary libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load the greyscale image
image_path = 'pexels-huy.jpg'
grayscale_image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
assert grayscale_image is not None, f"Could not load image '{image_path}' -- check the filename/path."

## AI colourisation

Runs the loaded greyscale image through the model and displays the original
next to the colourised result.

In [ ]:
import torch
from colorizers import siggraph17

# grayscale_image comes from the previous cell. Guard against running this
# cell on its own (e.g. after a kernel restart, or after editing image_path
# above without re-running it) with a clear error instead of a NameError or
# silently colourising a stale image left over from an earlier run.
assert 'grayscale_image' in dir() and grayscale_image is not None, (
    "grayscale_image is not defined -- run the previous cell (which loads the image) first."
)

# Load the pretrained colourisation model.
# Using siggraph17 instead of eccv16: eccv16 is the exact PyTorch port of the
# original "colorization_release_v2" Caffe weights, but it's known to produce
# desaturated, yellow/sepia-leaning results on a lot of photos. siggraph17 is
# the newer model from the same author/repo, trained with a better-balanced
# loss, and gives noticeably more vibrant, accurate colours on the same inputs.
colorizer = siggraph17(pretrained=True).eval()

# Resize target before running the model. siggraph17's skip connections only
# require each dimension to be a multiple of 8 -- it does NOT have to be
# exactly 256x256. Trade-off, roughly measured on a 2000x1000 photo (CPU):
#   256           -> ~2s,  matches the model's official demo/training size, coarsest colour detail
#   384-512       -> a few seconds, noticeably sharper colour, aspect ratio nearly untouched
#   None (native) -> ~96s for 2000x1000 (scales with pixel count), sharpest possible colour,
#                    but runs the model far outside the resolution it was trained on
MAX_DIM = 384  # try 256, 384, 512, or None

# Prepare image
img = grayscale_image.astype(np.float32) / 255.0
img_lab = cv2.cvtColor(cv2.merge([img, img, img]), cv2.COLOR_RGB2Lab)
L_orig = img_lab[:, :, 0]

def round_to_multiple_of_8(x):
    return max(8, round(x / 8) * 8)

h, w = L_orig.shape
if MAX_DIM is None:
    h_target, w_target = round_to_multiple_of_8(h), round_to_multiple_of_8(w)
else:
    scale = min(1.0, MAX_DIM / max(h, w))
    h_target = round_to_multiple_of_8(h * scale)
    w_target = round_to_multiple_of_8(w * scale)

L_input = cv2.resize(L_orig, (w_target, h_target))
L_tensor = torch.from_numpy(L_input)[None, None, :, :].float()

# Run colourisation (no user hints -> input_B/mask_B default to zero inside the model)
with torch.no_grad():
    ab_tensor = colorizer(L_tensor)
ab = ab_tensor[0].permute(1, 2, 0).numpy()
ab = cv2.resize(ab, (grayscale_image.shape[1], grayscale_image.shape[0]))

# Combine original-resolution L + upsampled ab and convert to RGB
result_lab = np.concatenate([img_lab[:, :, 0:1], ab], axis=2)
result_bgr = cv2.cvtColor(result_lab, cv2.COLOR_Lab2BGR)
result_rgb = (np.clip(result_bgr, 0, 1) * 255).astype(np.uint8)
result_rgb = cv2.cvtColor(result_rgb, cv2.COLOR_BGR2RGB)

# Display
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title('Greyscale')
plt.imshow(grayscale_image, cmap='gray')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.title('AI Colourised')
plt.imshow(result_rgb)
plt.axis('off')
plt.show()
